In [3]:
from pathlib import Path
import requests

OWNER = "behnamparsa"
REPO = "emulator-testing-operational-profiles"
RUN_ID = 23632521376

TOKENS_DIR = Path(r"C:\GitHub\Android-Mobile-Apps")
ENV_FILE = TOKENS_DIR / "All_Tokens.env"

def read_tokens_from_env_file(env_path):
    path = Path(env_path)
    if not path.exists():
        raise FileNotFoundError(f"Token file not found: {path}")

    text = path.read_text(encoding="utf-8", errors="replace")
    vals = {}

    for raw_line in text.splitlines():
        line = raw_line.strip()
        if not line or line.startswith("#") or "=" not in line:
            continue
        key, value = line.split("=", 1)
        vals[key.strip()] = value.strip().strip('"').strip("'")

    tokens = []
    for i in range(1, 8):
        tok = vals.get(f"GITHUB_TOKEN_{i}")
        if tok and tok not in tokens:
            tokens.append(tok)

    if not tokens:
        raise RuntimeError(f"No GITHUB_TOKEN_1..7 found in {path}")

    return tokens

tokens = read_tokens_from_env_file(ENV_FILE)
print(f"Using token file: {ENV_FILE}")
print(f"Loaded {len(tokens)} token(s)")

url = f"https://api.github.com/repos/{OWNER}/{REPO}/actions/runs/{RUN_ID}/logs"

for idx, token in enumerate(tokens, start=1):
    headers = {
        "Accept": "application/vnd.github+json",
        "Authorization": f"Bearer {token}",
        "X-GitHub-Api-Version": "2022-11-28",
    }

    try:
        resp = requests.get(url, headers=headers, allow_redirects=True, timeout=120)
        print(f"Token {idx}: HTTP {resp.status_code}")

        if resp.status_code == 200:
            out_file = Path.cwd() / f"run_{RUN_ID}_logs.zip"
            out_file.write_bytes(resp.content)
            print(f"Saved zip to: {out_file.resolve()}")
            break
        else:
            print(resp.text[:500])

    except Exception as e:
        print(f"Token {idx} failed with exception: {e}")
else:
    raise RuntimeError("All tokens failed.")

Using token file: C:\GitHub\Android-Mobile-Apps\All_Tokens.env
Loaded 7 token(s)
Token 1: HTTP 200
Saved zip to: C:\GitHub\Android-Mobile-Apps\ICST2026_RQ3_Extension\run_23632521376_logs.zip
